## Imports Libraries

In [ ]:
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import barcode
from barcode.writer import ImageWriter
from pathlib import Path
import gspread
from google.oauth2.service_account import Credentials
import numpy as np
from PIL import Image
from rembg import remove
import mediapipe as mp
import re
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

## **DEFINE THE FUNCTIONS NEEDED FOR IMAGE PROCESSING AND DRIVE DOWNLOADS**

In [ ]:
def extract_drive_file_id(url) -> str | None:
    """Pulls the file ID out of a Google Drive share link, handling the
    common URL formats Drive/Forms generates."""
    if not isinstance(url, str) or not url.strip():
        return None

    # Handles: drive.google.com/file/d/FILE_ID/view
    match = re.search(r'/d/([-\w]{25,})', url)
    if match:
        return match.group(1)

    # Handles: drive.google.com/open?id=FILE_ID  or  ...uc?id=FILE_ID
    match = re.search(r'[?&]id=([-\w]{25,})', url)
    if match:
        return match.group(1)

    # Fallback: just grab the longest ID-shaped token in the string
    match = re.search(r'[-\w]{25,}', url)
    return match.group(0) if match else None

In [ ]:
mp_face_detection = mp.solutions.face_detection

def is_background_white(img: Image.Image, tolerance=25, border_pct=0.05):
    """Sample a thin border strip around the photo and check how close
    the average color is to pure white."""
    img_rgb = img.convert("RGB")
    w, h = img_rgb.size
    arr = np.array(img_rgb)
    bw, bh = max(1, int(w * border_pct)), max(1, int(h * border_pct))

    border_pixels = np.concatenate([
        arr[:bh, :].reshape(-1, 3),
        arr[-bh:, :].reshape(-1, 3),
        arr[:, :bw].reshape(-1, 3),
        arr[:, -bw:].reshape(-1, 3),
    ])
    avg_color = border_pixels.mean(axis=0)
    distance_from_white = np.linalg.norm(np.array([255, 255, 255]) - avg_color)
    return distance_from_white < tolerance


def remove_and_whiten_background(img: Image.Image) -> Image.Image:
    """Cut the person out via rembg, then composite onto a solid white canvas."""
    img_rgba = remove(img.convert("RGBA"))
    white_bg = Image.new("RGBA", img_rgba.size, (255, 255, 255, 255))
    composited = Image.alpha_composite(white_bg, img_rgba)
    return composited.convert("RGB")


def detect_face_box(img: Image.Image):
    """Returns (x_min, y_min, x_max, y_max) in pixels, or None if no face found."""
    arr = np.array(img.convert("RGB"))
    with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as fd:
        results = fd.process(arr)
        if not results.detections:
            return None
        detection = max(results.detections, key=lambda d: d.score[0])
        box = detection.location_data.relative_bounding_box
        h, w, _ = arr.shape
        return (
            int(box.xmin * w), int(box.ymin * h),
            int((box.xmin + box.width) * w), int((box.ymin + box.height) * h),
        )


def recrop_on_face(img: Image.Image, face_box, target_aspect=190/250):
    """Recenter/recrop so the face follows standard ID-photo composition:
    face height ~50% of frame, with headroom above."""
    w, h = img.size
    x_min, y_min, x_max, y_max = face_box
    face_h = y_max - y_min
    face_cx, face_cy = (x_min + x_max) / 2, (y_min + y_max) / 2

    target_h = face_h / 0.5
    target_w = target_h * target_aspect
    crop_top = face_cy - target_h * 0.42
    crop_left = face_cx - target_w / 2
    crop_box = (int(crop_left), int(crop_top), int(crop_left + target_w), int(crop_top + target_h))

    # White-pad if the ideal crop extends past the original photo's edges
    canvas = Image.new("RGB", (crop_box[2] - crop_box[0], crop_box[3] - crop_box[1]), (255, 255, 255))
    src_box = (max(0, crop_box[0]), max(0, crop_box[1]), min(w, crop_box[2]), min(h, crop_box[3]))
    paste_pos = (max(0, -crop_box[0]), max(0, -crop_box[1]))
    canvas.paste(img.crop(src_box), paste_pos)
    return canvas

In [ ]:
def process_id_photo_from_image(img: Image.Image):
    flags = {"bg_fixed": False, "needs_review": False, "face_detected": False, "cached": False}

    if not is_background_white(img):
        img = remove_and_whiten_background(img)
        flags["bg_fixed"] = True

    face_box = detect_face_box(img)
    if face_box is None:
        flags["needs_review"] = True
    else:
        flags["face_detected"] = True
        w, _ = img.size
        face_cx = (face_box[0] + face_box[2]) / 2
        offset_pct = abs(face_cx - w / 2) / w
        if offset_pct > 0.08:
            img = recrop_on_face(img, face_box)
            flags["needs_review"] = True

    return img, flags


def process_id_photo(path):
    return process_id_photo_from_image(Image.open(path).convert("RGB"))

In [ ]:
def process_row(row):
    id_num = row["id_number"]
    photo_cache = PHOTOS_DIR / f"{id_num}.png"
    sig_cache = SIGNATURES_DIR / f"{id_num}.png"

    result = {"id_number": id_num, "photo": None, "photo_flags": {}, "signature": None, "signature_error": None}

    # --- Photo ---
    if photo_cache.exists():
        result["photo"] = Image.open(photo_cache)
        result["photo_flags"] = {"cached": True}
    else:
        file_id = extract_drive_file_id(row["id_picture_path"])
        if not file_id:
            result["photo_flags"] = {"error": "no valid photo URL/file ID"}
        else:
            try:
                raw_img = download_drive_image(file_id)
                processed_img, flags = process_id_photo_from_image(raw_img)
                processed_img.save(photo_cache)
                result["photo"] = processed_img
                result["photo_flags"] = flags
            except Exception as e:
                result["photo_flags"] = {"error": str(e)}

    # --- Signature ---
    if sig_cache.exists():
        result["signature"] = Image.open(sig_cache)
    else:
        sig_file_id = extract_drive_file_id(row["e_signature_path"])
        if not sig_file_id:
            result["signature_error"] = "no valid signature URL/file ID"
        else:
            try:
                raw_sig = download_drive_image(sig_file_id)
                processed_sig = process_signature_image(raw_sig)
                processed_sig.save(sig_cache)
                result["signature"] = processed_sig
            except Exception as e:
                result["signature_error"] = str(e)

    return result

In [ ]:
def process_signature_image(img: Image.Image, threshold=200) -> Image.Image:
    """Converts a signature photo/scan into a transparent PNG --
    light paper becomes transparent, dark ink stays opaque and solid black.
    Tightly cropped to just the ink, ready to overlay directly on the card."""
    gray = img.convert("L")
    arr = np.array(gray)

    # Pixels darker than the threshold become opaque; lighter ones transparent
    alpha = np.where(arr < threshold, 255, 0).astype(np.uint8)

    black_layer = Image.new("RGBA", img.size, (20, 20, 20, 255))
    transparent_layer = Image.new("RGBA", img.size, (0, 0, 0, 0))
    signature_rgba = Image.composite(black_layer, transparent_layer, Image.fromarray(alpha))

    bbox = signature_rgba.getbbox()   # crop tightly to just the ink, no wasted white space
    if bbox:
        signature_rgba = signature_rgba.crop(bbox)

    return signature_rgba

def process_signature_row(row):
    id_num = row["id_number"]
    photo_cache = PHOTOS_DIR / f"{id_num}.png"
    sig_cache = SIGNATURES_DIR / f"{id_num}.png"

    result = {"id_number": id_num, "photo": None, "photo_flags": {}, "signature": None, "signature_error": None}

    # --- Photo (same as before) ---
    if photo_cache.exists():
        result["photo"] = Image.open(photo_cache)
        result["photo_flags"] = {"cached": True}
    else:
        file_id = extract_drive_file_id(row["id_picture_path"])
        if file_id:
            try:
                raw_img = download_drive_image(file_id)
                processed_img, flags = process_id_photo_from_image(raw_img)
                processed_img.save(photo_cache)
                result["photo"] = processed_img
                result["photo_flags"] = flags
            except Exception as e:
                result["photo_flags"] = {"error": str(e)}

    # --- Signature (new) ---
    if sig_cache.exists():
        result["signature"] = Image.open(sig_cache)
    else:
        sig_file_id = extract_drive_file_id(row["e_signature_path"])
        if sig_file_id:
            try:
                raw_sig = download_drive_image(sig_file_id)
                processed_sig = process_signature_image(raw_sig)
                processed_sig.save(sig_cache)   # PNG preserves transparency
                result["signature"] = processed_sig
            except Exception as e:
                result["signature_error"] = str(e)

    return result

In [ ]:
def cover_resize(img: Image.Image, target_w: int, target_h: int) -> Image.Image:
    """Resize to completely fill (target_w, target_h), cropping any overflow --
    unlike .thumbnail(), this guarantees every output is exactly the same size."""
    img = img.convert("RGB")
    src_w, src_h = img.size
    scale = max(target_w / src_w, target_h / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    resized = img.resize((new_w, new_h), Image.LANCZOS)

    left = (new_w - target_w) // 2
    top = (new_h - target_h) // 2
    return resized.crop((left, top, left + target_w, top + target_h))

In [ ]:
def build_preview_sheet(all_results, photo_w=90, photo_h=115, sig_w=90, sig_h=115, label_font_size=11):
    ids = list(all_results.keys())
    cols = 4
    thumb_size = photo_w + sig_w
    rows = math.ceil(len(ids) / cols) if ids else 1
    cell_h = photo_h + 70

    sheet = Image.new("RGB", (cols * thumb_size, rows * cell_h), (235, 235, 235))
    draw = ImageDraw.Draw(sheet)
    label_font = ImageFont.truetype("C:/Windows/Fonts/arialbd.ttf", label_font_size)
    small_font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", label_font_size - 1)

    for i, id_num in enumerate(ids):
        result = all_results[id_num]
        col, row = i % cols, i // cols
        x, y = col * thumb_size, row * cell_h

        # --- Photo: fixed size, cropped to fill ---
        photo = result.get("photo")
        if photo:
            thumb = cover_resize(photo, photo_w, photo_h)
            sheet.paste(thumb, (x, y))
        else:
            draw.rectangle([x, y, x + photo_w, y + photo_h], outline=(200, 0, 0), width=2)
            draw.text((x + 6, y + photo_h // 2), "NO PHOTO", font=small_font, fill=(200, 0, 0))

        # --- Signature: same fixed size, but "contain" not "cover" so strokes never get cropped off ---
        sig = result.get("signature")
        sig_bg = Image.new("RGB", (sig_w, sig_h), (255, 255, 255))
        if sig:
            sig_thumb = sig.copy()
            sig_thumb.thumbnail((sig_w - 8, sig_h - 8))
            paste_x = (sig_w - sig_thumb.width) // 2
            paste_y = (sig_h - sig_thumb.height) // 2
            sig_bg.paste(sig_thumb, (paste_x, paste_y), sig_thumb if sig_thumb.mode == "RGBA" else None)
        else:
            d2 = ImageDraw.Draw(sig_bg)
            d2.rectangle([0, 0, sig_w - 1, sig_h - 1], outline=(200, 0, 0), width=2)
            d2.text((6, sig_h // 2), "NO SIG", font=small_font, fill=(200, 0, 0))
        sheet.paste(sig_bg, (x + photo_w, y))

        # --- Labels ---
        draw.text((x + 4, y + photo_h + 4), id_num, font=label_font, fill=(20, 20, 20))
        status_lines = []
        if result["photo_flags"].get("needs_review"):
            status_lines.append("photo: REVIEW")
        if result["photo_flags"].get("error"):
            status_lines.append(f"photo err: {result['photo_flags']['error'][:20]}")
        if result["signature_error"]:
            status_lines.append(f"sig err: {result['signature_error'][:20]}")
        sy = y + photo_h + 22
        for line in status_lines:
            draw.text((x + 4, sy), line, font=small_font, fill=(180, 0, 0))
            sy += 14

    return sheet

## SETUP PATHS

In [ ]:
BASE_DIR = Path.cwd()
TEMPLATE_DIR = BASE_DIR / "templates"
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output" / "cards"
FONT_DIR = BASE_DIR / "fonts"
PHOTOS_DIR = BASE_DIR / "photos"
CREDS_PATH = BASE_DIR / "credentials" / "service_account.json"
SIGNATURES_DIR = BASE_DIR / "signatures"
SIGNATURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.auth.transport.requests import AuthorizedSession
from io import BytesIO

SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly", "https://www.googleapis.com/auth/drive.readonly",]
creds = Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)

# Reuses the same `creds` object from Stage 2b -- no new auth setup needed
authed_session = AuthorizedSession(creds)

def download_drive_image(file_id: str) -> Image.Image:
    url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media"
    response = authed_session.get(url)
    response.raise_for_status()   # raises a clear error if the file ID is wrong or permission denied
    return Image.open(BytesIO(response.content)).convert("RGB")

## IMPORTS THE IMAGE TEMPLATES

In [ ]:
front_template = Image.open(TEMPLATE_DIR / "front_template.png")
back_template = Image.open(TEMPLATE_DIR / "back_template.png")

## IMPORT THE RESPONSES FROM GOOGLE SHEET

In [ ]:
gc = gspread.authorize(creds)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1_GKJPfENYbBNBoz11J-eYmQquKm1TTCxu63ChhnLwDM/edit"
sheet = gc.open_by_url(SHEET_URL).sheet1

records = sheet.get_all_records()
df = pd.DataFrame(records)

print(f"Loaded {len(df)} responses")
df.head()

In [ ]:
print(df.columns.tolist())

## **RENAME THE COLUMNS FETCHED IN GOOGLE FORM RESPONSE**

In [ ]:
df = df.rename(columns={
    "Timestamp": "date_submitted",
    "ID NUMBER": "id_number",
    "FULL NAME": "full_name",
    "POSITION": "position",
    "HOME ADDRESS": "address",
    "TIN NUMBER (if any)": "tin",
    "GSIS NUMBER (if any)": "gsis",
    "BLOOD TYPE": "blood_type",
    "EMERGENCY CONTACT PERSON": "emergency_contact_person",
    "EMERGENCY CONTACT NUMBER": "emergency_contact_number",
    "E-SIGNATURE": "e_signature_path",
    "ID PICTURE": "id_picture_path",
    "EMPLOYEE TYPE": "employee_type",
})

In [ ]:
df.head()

## **REPLACE AND VALIDATES ALL MISSING FIELDS**

In [ ]:
PLACEHOLDER_ID = "EMB-000-0000"

# TRIM THE ID NUMBER AND REPLACE IT IF MISSING
df["id_number"] = df["id_number"].astype(str).str.strip()
df["id_number"] = df["id_number"].replace(["", "nan", "None"], PLACEHOLDER_ID)

# CATCHES AND REPLACES ACTUAL NAN/NON VALUES
df = df.fillna("N/A")
df = df.replace(r'^\s*$', "N/A", regex=True)

# TRANSFORM ALL OF THE INPUTS TO UPPERCASE

exclude_cols = ["e_signature_path", "id_picture_path"]

# Confirm the exclude columns actually exist before proceeding --
# catches typos/mismatches early instead of silently skipping nothing
missing_exclude_cols = [c for c in exclude_cols if c not in df.columns]
if missing_exclude_cols:
    print(f"WARNING: these exclude columns don't exist in df: {missing_exclude_cols}")
    print(f"Actual columns: {df.columns.tolist()}")

cols_to_upper = [col for col in df.columns if col not in exclude_cols]

# TRANSFORMS THE INPUTS 
for col in cols_to_upper:
    if(df[col].dtype == 'str'):
       df[col] = df[col].astype(str).str.upper()

df

In [ ]:
LAST_RUN_FILE = DATA_DIR / "last_run.json"
MAX_WORKERS = 8   # concurrent downloads -- see note below on why not higher

# --- Filter to only rows submitted since the last successful run ---
df["date_submitted"] = pd.to_datetime(df["date_submitted"])
if LAST_RUN_FILE.exists():
    with open(LAST_RUN_FILE) as f:
        last_ts = pd.to_datetime(json.load(f)["last_processed_timestamp"])
    df_to_process = df[df["date_submitted"] > last_ts].copy()
    print(f"Last run: {last_ts} -- processing {len(df_to_process)} new submission(s)")
else:
    df_to_process = df.copy()
    print(f"No previous run found -- processing all {len(df_to_process)} row(s)")

# --- Run all rows concurrently instead of one at a time ---
all_results = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_row, row): row["id_number"] for _, row in df_to_process.iterrows()}
    for future in as_completed(futures):
        result = future.result()
        all_results[result["id_number"]] = result
        print(f"{result['id_number']}: photo={result['photo_flags']}, sig_error={result['signature_error']}")

# --- Save a marker so the next run only picks up what's new after this ---
if len(df_to_process) > 0:
    with open(LAST_RUN_FILE, "w") as f:
        json.dump({"last_processed_timestamp": df_to_process["date_submitted"].max().isoformat()}, f)

print(f"\nDone. Processed {len(all_results)} row(s).")